In [1]:
print("ibtisam")

ibtisam


In [ ]:
! uv pip install langchain openai langchain_groq tiktoken rapidocr-onnxruntime python-dotenv langchain-community langchain-google-genai langchain_openai 


Using Python 3.13.9 environment at: C:\Users\PC\OneDrive\Desktop\LLMOPS_PRACTICE\.venv
  × No solution found when resolving dependencies:
  ╰─▶ Because langchain-schema was not found in the package registry and you
      require langchain-schema, we can conclude that your requirements are
      unsatisfiable.


In [3]:
import os 
from dotenv import load_dotenv
load_dotenv()


os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ['GOOGLE_API_KEY'] = os.getenv("GOOGLE_API_KEY")

# DATA INGESTION (IN VECTORSTORE)

In [4]:
from langchain_community.document_loaders import TextLoader

In [5]:
loader = TextLoader("C:/Users/PC/OneDrive/Desktop/LLMOPS_PRACTICE/data/Hitler.txt" , encoding = 'utf-8' , autodetect_encoding = True)

In [6]:
documents = loader.load()

In [7]:
documents[0].metadata

{'source': 'C:/Users/PC/OneDrive/Desktop/LLMOPS_PRACTICE/data/Hitler.txt'}

In [8]:
documents

[Document(metadata={'source': 'C:/Users/PC/OneDrive/Desktop/LLMOPS_PRACTICE/data/Hitler.txt'}, page_content="Sure, here’s a detailed report on Adolf Hitler. Let me know if you want this tailored for a specific audience—academic, school project, historical analysis, or any other purpose.\n\n---\n\n## **Detailed Report on Adolf Hitler**\n\n### **1. Introduction**\nAdolf Hitler (1889–1945) was a German politician, dictator, and the leader of the National Socialist German Workers' Party (Nazi Party). He is known as one of history’s most infamous figures due to his central role in initiating World War II and orchestrating the Holocaust, which led to the deaths of millions. His life and actions had a profound and devastating impact on the 20th century.\n\n---\n\n### **2. Early Life and Background**\n- **Born:** April 20, 1889, in Braunau am Inn, Austria-Hungary (modern-day Austria).\n- **Parents:** Alois Hitler (father), Klara Hitler (mother).\n- Hitler had a troubled relationship with his f

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [10]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 100 , chunk_overlap = 10)
texts = text_splitter.split_documents(documents)

In [11]:
texts

[Document(metadata={'source': 'C:/Users/PC/OneDrive/Desktop/LLMOPS_PRACTICE/data/Hitler.txt'}, page_content='Sure, here’s a detailed report on Adolf Hitler. Let me know if you want this tailored for a specific'),
 Document(metadata={'source': 'C:/Users/PC/OneDrive/Desktop/LLMOPS_PRACTICE/data/Hitler.txt'}, page_content='specific audience—academic, school project, historical analysis, or any other purpose.'),
 Document(metadata={'source': 'C:/Users/PC/OneDrive/Desktop/LLMOPS_PRACTICE/data/Hitler.txt'}, page_content='---\n\n## **Detailed Report on Adolf Hitler**'),
 Document(metadata={'source': 'C:/Users/PC/OneDrive/Desktop/LLMOPS_PRACTICE/data/Hitler.txt'}, page_content='### **1. Introduction**'),
 Document(metadata={'source': 'C:/Users/PC/OneDrive/Desktop/LLMOPS_PRACTICE/data/Hitler.txt'}, page_content='Adolf Hitler (1889–1945) was a German politician, dictator, and the leader of the National'),
 Document(metadata={'source': 'C:/Users/PC/OneDrive/Desktop/LLMOPS_PRACTICE/data/Hitler.txt

In [12]:
! uv pip install faiss-cpu

Using Python 3.13.9 environment at: C:\Users\PC\OneDrive\Desktop\LLMOPS_PRACTICE\.venv
Audited 1 package in 8ms


In [13]:

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

In [14]:
openai_embeddings = OpenAIEmbeddings()
google_embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [15]:
vector_store = FAISS.from_documents(texts , google_embeddings)

In [16]:
vector_store

In [24]:
retriever = vector_store.as_retriever()

## DATA RETRIEVAL

In [26]:
query = "What are the Nazi Rule and Policies"

docs = vector_store.similarity_search(query , k = 4)

In [29]:
docs[3].page_content

'- **Eugenics and Racial Purity:** Programs like forced sterilization and euthanasia of disabled'

In [19]:
from langchain_core.prompts import ChatPromptTemplate

template = """"
You are helpful assistant for question-answering tasks 
use the retrieved context to answer the question if you dont know the answer just say i dont know the answer 
use maximum 10 sentence and keep the answer consice
Question:{question}
Context: {context}
Answer:


"""

In [20]:
prompt = ChatPromptTemplate.from_template(template)

In [21]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='"\nYou are helpful assistant for question-answering tasks \nuse the retrieved context to answer the question if you dont know the answer just say i dont know the answer \nuse maximum 10 sentence and keep the answer consice\nQuestion:{question}\nContext: {context}\nAnswer:\n\n\n'), additional_kwargs={})])

In [32]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [42]:
from langchain_groq import ChatGroq

llm = ChatGroq(model= "llama-3.1-8b-instant")

In [43]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever , "question" : RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

In [44]:
rag_chain.invoke("What are the Nazi Rule and Policies")

'The Nazi Rule and Policies included:\n\n- Anti-Semitic Laws: The Nuremberg Laws (1935) stripped Jews of citizenship and civil rights.\n- Propaganda and censorship, using the secret police (Gestapo) to control and suppress opposition.\n- Eugenics and Racial Purity: Programs like forced sterilization and euthanasia of disabled individuals were implemented.\nThese policies aimed to promote racial purity and eliminate perceived threats to the Aryan race.'